# Member C — Project Part 2: Label Foundation, Enrichment, and Leakage-Safe Data Design

This notebook is the **primary executable notebook for Project Part 2**.

The overall project is organized as:

1. **Part 1 — Scraping and cleaning**: acquire Ninisite posts, repair metadata, preserve raw text, and construct the cleaned website-scale corpus.
2. **Part 2 — This notebook**: reconstruct the final stress labels, attach them safely to the cleaned corpus, prevent author/thread/duplicate leakage, and create the exact train/validation/test/embargo roles and OOF folds used by the models.
3. **Part 3 — Tabular and Transformer modeling**: Member 1 trains the structured CatBoost branch and Member 2 trains the Persian ParsBERT branch.
4. **Part 4 — Fusion and final evaluation**: combine the two leakage-safe base-model outputs, calibrate thresholds, evaluate the locked test, explain the model, and demonstrate monitoring/active-learning utilities.

The source files in this package still use the historical names **Phase 1, Phase 2, Phase 3, and Phase 3.1**. Those phase names are intentionally preserved inside the code and artifact filenames for provenance. In the final project organization, all of them belong to **Part 2**.

## Why Part 2 is scientifically important

Model performance is meaningful only if the labels and evaluation split are trustworthy. In a forum dataset, ordinary row-random splitting is unsafe because posts can be related through the same author, the same conversation thread, or exact duplicate content. Similarly, annotation rounds can contain repeated post IDs, different annotator provenance, and weaker single-rater labels.

Part 2 therefore solves four methodological problems **before any final model is trained**:

1. **Label reconstruction:** combine multiple historical annotation rounds into one canonical stress target per post while preserving all provenance.
2. **Duplicate-safe enrichment:** join labels back to the cleaned website data without silently choosing the wrong row when IDs are duplicated.
3. **Leakage-safe grouping:** connect rows that share author/thread/duplicate-content relations and keep the entire connected component on one side of the evaluation boundary.
4. **Modeling-role finalization:** separate training, official validation, locked test, and embargo rows; then construct grouped OOF folds and training weights.

This is not bookkeeping. It is part of the experimental design. If any of these steps is wrong, a later CatBoost/Transformer/fusion score can look impressive while measuring leakage rather than generalization.

## Key outputs of Part 2

The final contract is:

- **5,615 enriched labeled posts**
- **4,226 train**
- **452 validation**
- **453 locked test**
- **484 embargo**
- five grouped OOF folds of **844 / 844 / 848 / 847 / 843** training rows

The evaluation rows are deliberately **safety-enriched** and therefore must not be used to estimate the natural prevalence of stress on Ninisite.

## Reproducibility note

A true historical Phase 2 rerun needs the large cleaned website CSV. If that external file is unavailable, this notebook restores the accepted frozen Phase 2 enrichment and still reruns label reconstruction, split construction, OOF-fold assignment, weights, and handoff generation. Cached reproduction is transparent: the notebook prints which mode was used instead of pretending the raw extraction was rerun.


## 0. Execution parameters — controlling reproducibility without changing the experiment

This cell defines **execution behavior**, not model hyperparameters.

### Why keep frozen inputs separate from fresh outputs?

The notebook writes every new artifact under `notebook_run/`. The distributed reference artifacts are never overwritten. This is important because reproducibility requires comparing a fresh run against a known reference rather than destroying the reference during execution.

### Important parameters

- `USE_FROZEN_PHASE2_IF_RAW_MISSING=True`: if the large cleaned corpus is not present, restore the exact accepted Phase 2 enrichment. This keeps the notebook executable while clearly recording that raw-data streaming was not repeated.
- `RESET_NOTEBOOK_RUN=True`: begin from a clean output directory so stale artifacts cannot be mistaken for newly generated results.
- `PHASE3_RESTARTS=300`: the component-to-split assignment is a constrained combinatorial problem. Multiple randomized greedy restarts search for a better-balanced assignment while preserving whole connected components.
- `PHASE3_SEED=20260804`: fixes the pseudo-random sequence used by the split optimizer and later grouped folds, making the accepted split reproducible.

### Why a seed is necessary but not sufficient

A seed controls intended randomness. Reproducibility also depends on frozen inputs, fixed code, stable IDs, package versions, and explicit assertions. That is why this notebook uses both deterministic configuration **and** artifact-level validation.


In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import hashlib
import gzip
import pandas as pd
from IPython.display import display, Markdown

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current.parent, current.parent.parent]
    for candidate in candidates:
        if (candidate / "src/phase1_reconstruct_labels.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the Phase 1–3 project root. "
        "Open this notebook from its packaged notebooks directory."
    )

ROOT = find_project_root()
RUN_ROOT = ROOT / "notebook_run"

RAW_360K_ENV = str(ROOT / "inputs" / "ninisite_full_dataset.csv")
RAW_360K_PATH = Path(RAW_360K_ENV)

USE_FROZEN_PHASE2_IF_RAW_MISSING = True
RESET_NOTEBOOK_RUN = True
PHASE3_RESTARTS = 300
PHASE3_SEED = 20260804
PYTHON = sys.executable

if RESET_NOTEBOOK_RUN and RUN_ROOT.exists():
    shutil.rmtree(RUN_ROOT)

for folder in ["phase1", "phase2", "phase3", "phase3_1"]:
    (RUN_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Notebook output root:", RUN_ROOT)
print("Python:", PYTHON)
print("Raw 360k path:", RAW_360K_PATH)
print("Raw 360k available:", bool(RAW_360K_PATH and RAW_360K_PATH.exists()))

Project root: /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3
Notebook output root: /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run
Python: /opt/pyvenv/bin/python
Raw 360k path: None
Raw 360k available: False


**How to interpret the output**

The printed project root confirms that the notebook found the packaged `src/` implementation rather than some unrelated local copy. The raw-data flag tells us whether duplicate-safe enrichment will be recomputed from the large website file or restored from the accepted frozen artifact.

This distinction should be reported honestly: *reproducing downstream results from a frozen accepted enrichment is not the same operation as re-streaming the original 360k-row file*, even though both paths feed the same accepted Part 2 contract.


## 0.1 Command helper and input validation — fail early, not silently

The notebook deliberately calls the substantial algorithms as normal Python modules in `src/` instead of hiding them in one very long notebook cell.

### Why this design?

- **Testability:** standalone modules can be unit-tested independently.
- **Reusability:** the same logic can be called from PowerShell/terminal, not only Jupyter.
- **Auditability:** each stage has explicit input/output files, which makes hidden notebook state less likely.
- **Visible failure:** `run_command()` captures stdout/stderr and raises when a command returns an unexpected status.

Before label reconstruction, the notebook verifies that all historical annotation exports and the historical fusion index are present. A missing annotation file would change the canonical dataset, so the correct behavior is to stop rather than silently continue with fewer labels.

This is a general reproducibility principle: **validate the data contract before running the algorithm**.


In [2]:
def run_command(command, cwd=ROOT, allowed_returncodes=(0,)):
    command = [str(value) for value in command]
    print("\n$", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode not in allowed_returncodes:
        raise RuntimeError(
            f"Command failed with return code {completed.returncode}: {command}"
        )
    return completed

required_inputs = [
    "original_person1_2000.csv",
    "original_person2_2000.csv",
    "historical_fusion_index.csv",
    "active_candidates_3000.csv",
    "active_person1_1000.csv",
    "active_person2_3000.csv",
    "active_historical_final_3000.csv",
]
missing_inputs = [
    name for name in required_inputs if not (ROOT / "inputs" / name).exists()
]
if missing_inputs:
    raise FileNotFoundError(f"Missing Phase 1 inputs: {missing_inputs}")

print("All Phase 1 source files are present.")
display(pd.DataFrame({
    "input_file": required_inputs,
    "size_bytes": [(ROOT / "inputs" / name).stat().st_size for name in required_inputs],
}))

All Phase 1 source files are present.


,input_file,size_bytes
0,original_person1_2000.csv,627005
1,original_person2_2000.csv,627051
2,historical_fusion_index.csv,70580
3,active_candidates_3000.csv,1560898
4,active_person1_1000.csv,491461
5,active_person2_3000.csv,1503339
6,active_historical_final_3000.csv,1472838


**How to interpret the output**

Every required annotation source should appear with a nonzero file size. Their presence does not prove that the labels are scientifically perfect, but it proves that the reconstruction is using the intended historical sources.

The next stage will preserve annotation provenance because dual-annotated, single-rater, and calibrated labels do not have the same evidential strength.


# Historical Phase 1 — Reconstruct one canonical stress label per post

This stage resolves the fact that the same post may appear in multiple annotation rounds and may have different annotation provenance.

## 1. Dual-annotator consensus

For rows with both historical annotators, the project used a weighted consensus in which Person 1 received weight 1.7 and Person 2 weight 1.0:

\[
y_{\text{consensus}} = \frac{1.7\,y_1 + 1.0\,y_2}{2.7}
\]

A weighted mean is still a continuous target, which is useful because later models are regressors rather than direct four-class classifiers.

The 1.7:1 ratio is a **historical project choice**, not a universal clinical weighting. The reproducibility objective is to reconstruct the target that was actually used, while retaining enough provenance to audit weaker labels separately.

## 2. Calibrating Person 2-only active-learning labels

The active-learning history contains 2,000 rows labeled only by Person 2. Simply treating those values as equivalent to dual consensus would ignore systematic annotator-scale differences.

The project therefore uses the 1,000 dual-labeled active-learning overlap to learn a linear mapping from Person 2's score to Person 1's scale. Conceptually:

\[
\widehat{y_1} = a\,y_2 + b
\]

Then \(\widehat{y_1}\) and \(y_2\) are combined using the same historical 1.7:1 consensus rule. The calibration is checked with cross-validation on the overlap so that the chosen mapping is evaluated on held-out annotator pairs rather than only on the data used to fit it.

This does **not** turn a single-rater label into clinical ground truth. It creates a lower-confidence training label whose provenance remains explicit.

## 3. Canonicalization and provenance

A canonical table must contain one row per `unique_post_id`. If an ID appears in more than one annotation round, the pipeline chooses the canonical observation according to the historical provenance rules instead of double-counting the post. At the same time, `annotation_history.csv.gz` retains every observed or reconstructed annotation event.

This two-table design separates:

- **canonical modeling target** — one row per post;
- **audit history** — every annotation event and decision.

Exact duplicate content is also identified so that later split construction can keep repeated text together.

## 4. Why `clinical_class` is preserved rather than recomputed

The frozen `clinical_class` field is the authoritative evaluation class. Later stages do not silently regenerate classes from rounded continuous values. This avoids changing historical boundary cases during reproduction.

The operational class convention used by the final project is Low ≤3, Moderate (3,5], High (5,7], Very High >7, but the frozen class column remains the evaluation authority for the accepted dataset.


In [3]:
phase1_output = RUN_ROOT / "phase1"

run_command([
    PYTHON,
    ROOT / "src/phase1_reconstruct_labels.py",
    "--input-dir", ROOT / "inputs",
    "--output-dir", phase1_output,
])

phase1_report = json.loads(
    (phase1_output / "phase1_report.json").read_text(encoding="utf-8")
)
canonical = pd.read_csv(
    phase1_output / "canonical_labels.csv.gz",
    dtype={"unique_post_id": "string"},
)
annotation_history = pd.read_csv(
    phase1_output / "annotation_history.csv.gz",
    dtype={"unique_post_id": "string"},
)

display(Markdown("### Phase 1 execution report"))
display(pd.json_normalize(phase1_report))
display(Markdown("### Canonical class distribution"))
display(
    canonical["clinical_class"]
    .value_counts()
    .reindex(["Low", "Moderate", "High", "Very High"])
    .rename_axis("clinical_class")
    .reset_index(name="rows")
)
print("Canonical rows:", len(canonical))
print("Unique post IDs:", canonical["unique_post_id"].nunique())
print("Annotation-history rows:", len(annotation_history))
print("Duplicate canonical IDs:", canonical["unique_post_id"].duplicated().sum())


$ /opt/pyvenv/bin/python /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/src/phase1_reconstruct_labels.py --input-dir /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/inputs --output-dir /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase1


{
  "original_dual_rows": 2000,
  "historical_fusion_rows": 2229,
  "original_dual_in_historical_fusion": 1607,
  "original_dual_removed_by_historical_undersampling": 393,
  "recovered_original_p1_high_selected": 622,
  "active_rows": 3000,
  "active_dual_rows": 1000,
  "active_p2_calibrated_rows": 2000,
  "duplicate_ids_across_labeling_rounds": 5,
  "canonical_unique_posts": 5617,
  "annotation_history_rows": 10622
}
Source counts: {'active_dual': 1000, 'active_p2_calibrated': 1995, 'original_p1_high_selected': 622, 'original_dual': 2000}
Class counts: {'Moderate': 1222, 'Low': 3029, 'High': 974, 'Very High': 392}
Wrote outputs to /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase1

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup
  File "/tmp/tmp.yTcnQsZYi

### Phase 1 execution report

,status,class_thresholds,person1_weight,person2_weight,duplicate_id_values,exact_duplicate_content_groups,exact_duplicate_content_rows,default_label_weights_are_provisional,phase2_required,phase3_blocker,counts.original_dual_rows,counts.historical_fusion_rows,counts.original_dual_in_historical_fusion,counts.original_dual_removed_by_historical_undersampling,counts.recovered_original_p1_high_selected,counts.active_rows,counts.active_dual_rows,counts.active_p2_calibrated_rows,counts.duplicate_ids_across_labeling_rounds,counts.canonical_unique_posts,counts.annotation_history_rows,canonical_source_counts.active_dual,canonical_source_counts.active_p2_calibrated,canonical_source_counts.original_p1_high_selected,canonical_source_counts.original_dual,canonical_class_counts.Moderate,canonical_class_counts.Low,canonical_class_counts.High,canonical_class_counts.Very High,source_by_class_counts.active_dual.Moderate,source_by_class_counts.active_dual.Low,source_by_class_counts.active_dual.High,source_by_class_counts.active_dual.Very High,source_by_class_counts.active_p2_calibrated.Low,source_by_class_counts.active_p2_calibrated.Moderate,source_by_class_counts.active_p2_calibrated.Very High,source_by_class_counts.active_p2_calibrated.High,source_by_class_counts.original_p1_high_selected.High,source_by_class_counts.original_p1_high_selected.Moderate,source_by_class_counts.original_p1_high_selected.Very High,source_by_class_counts.original_dual.Low,source_by_class_counts.original_dual.High,source_by_class_counts.original_dual.Moderate,source_by_class_counts.original_dual.Very High,active_calibration.method_reproducing_historical_labels,active_calibration.person2_to_person1_slope,active_calibration.person2_to_person1_intercept,active_calibration.equivalent_person2_to_consensus_slope,active_calibration.equivalent_person2_to_consensus_intercept,active_calibration.cv_metrics.identity.mae_to_consensus,active_calibration.cv_metrics.identity.rmse_to_consensus,active_calibration.cv_metrics.identity.mae_to_person1,active_calibration.cv_metrics.mean_shift.mae_to_consensus,active_calibration.cv_metrics.mean_shift.rmse_to_consensus,active_calibration.cv_metrics.mean_shift.mae_to_person1,active_calibration.cv_metrics.linear_to_consensus.mae_to_consensus,active_calibration.cv_metrics.linear_to_consensus.rmse_to_consensus,active_calibration.cv_metrics.linear_to_consensus.mae_to_person1,active_calibration.cv_metrics.linear_p2_to_p1_then_weighted.mae_to_consensus,active_calibration.cv_metrics.linear_p2_to_p1_then_weighted.rmse_to_consensus,active_calibration.cv_metrics.linear_p2_to_p1_then_weighted.mae_to_person1,active_calibration.cv_metrics.isotonic_to_consensus.mae_to_consensus,active_calibration.cv_metrics.isotonic_to_consensus.rmse_to_consensus,active_calibration.cv_metrics.isotonic_to_consensus.mae_to_person1,active_calibration.cv_metrics.ordinal_lookup.mae_to_consensus,active_calibration.cv_metrics.ordinal_lookup.rmse_to_consensus,active_calibration.cv_metrics.ordinal_lookup.mae_to_person1
0,phase1_complete,"[3.0, 5.0, 7.0]",1.7,1.0,"[308300129, 337616795, 380905300, 394085325, t...",14,30,True,Run phase2_extract_enriched.py locally against...,Final author/thread connected-component split ...,2000,2229,1607,393,622,3000,1000,2000,5,5617,10622,1000,1995,622,2000,1222,3029,974,392,190,541,178,91,1058,514,117,306,251,263,108,1430,239,255,76,"OLS Person2->Person1 on 1000 overlaps, then we...",0.645927,0.503758,0.777065,0.317181,0.69763,0.996726,1.108,0.664146,0.824393,1.074516,0.391315,0.534587,0.695693,0.391315,0.534587,0.695693,0.39778,0.527445,0.699129,0.39778,0.527445,0.699129


### Canonical class distribution

,clinical_class,rows
0,Low,3029
1,Moderate,1222
2,High,974
3,Very High,392


Canonical rows: 5617
Unique post IDs: 5617
Annotation-history rows: 10622
Duplicate canonical IDs: 0


**What a successful Phase 1 result means**

The important invariant is **one canonical row per post ID**, not merely that the script finishes.

Expected outputs include:

- `canonical_labels.csv.gz` — one modeling target per post;
- `annotation_history.csv.gz` — complete annotation provenance;
- duplicate-ID/content audits;
- `phase1_report.json` — calibration and reconstruction statistics.

The canonical dataset contains **5,617 unique post IDs before website enrichment**. Some of these will later fail safe enrichment because the website export contains ambiguous duplicate-ID matches; those rows are intentionally handled in Phase 2 instead of being guessed here.


## Historical Phase 1.1 — Inspect calibration and label provenance

This cell does not train a predictive stress model. It audits the quality and composition of the reconstructed supervision.

### Why inspect calibration parameters?

A linear calibration has two interpretable terms:

- **slope:** whether Person 2's score scale expands/contracts relative to Person 1;
- **intercept:** whether there is a systematic level shift.

Cross-validated error on the dual-labeled overlap tells us how well the calibration predicts an unseen consensus pair. That is more informative than reporting only in-sample fit.

### Why summarize label sources and confidence tiers?

The final training set intentionally combines labels with different provenance strengths. Stronger dual labels can be used for official validation/test, while weaker single-rater/calibrated labels can enlarge training coverage with lower weight.

This is a form of **provenance-aware supervision**: we do not pretend every target has identical certainty.


In [4]:
calibration = phase1_report.get("active_calibration", {})
display(pd.DataFrame([calibration]))

source_summary = (
    canonical.groupby(["label_source", "confidence_tier"], dropna=False)
    .agg(
        rows=("unique_post_id", "size"),
        mean_stress=("final_stress", "mean"),
        minimum_stress=("final_stress", "min"),
        maximum_stress=("final_stress", "max"),
    )
    .reset_index()
    .sort_values(["label_source", "confidence_tier"])
)
display(source_summary)

duplicate_summary = pd.DataFrame({
    "audit": [
        "IDs repeated across annotation rounds",
        "Rows in exact duplicate-content groups",
    ],
    "rows": [
        int(canonical["duplicate_id_across_rounds"].fillna(False).astype(bool).sum()),
        int(canonical["exact_duplicate_content"].fillna(False).astype(bool).sum()),
    ],
})
display(duplicate_summary)

,method_reproducing_historical_labels,person2_to_person1_slope,person2_to_person1_intercept,equivalent_person2_to_consensus_slope,equivalent_person2_to_consensus_intercept,cv_metrics
0,"OLS Person2->Person1 on 1000 overlaps, then we...",0.645927,0.503758,0.777065,0.317181,{'identity': {'mae_to_consensus': 0.6976296296...


,label_source,confidence_tier,rows,mean_stress,minimum_stress,maximum_stress
0,active_dual,high,1000,3.380370,1.000000,9.370370
1,active_p2_calibrated,medium,1995,3.257952,1.094246,8.087829
2,original_dual,high,2000,2.431574,1.000000,9.370370
3,original_p1_high_selected,medium_high,622,5.012862,3.000000,10.000000


,audit,rows
0,IDs repeated across annotation rounds,5
1,Rows in exact duplicate-content groups,30


**How to interpret this audit**

Dual-annotation rows are the strongest internal evidence and are the only label sources used as official validation/test ground truth in the final modeling roles.

Calibrated single-rater rows remain useful for training because they expand coverage, especially after active learning, but their lower confidence is encoded later through `annotation_weight`.

The duplicate summaries also matter for leakage control: two rows with identical text should not be allowed to land in different folds merely because their IDs differ.


# Historical Phase 2 — Duplicate-safe enrichment from the cleaned website corpus

The canonical label table contains target/provenance fields, but Member 1 and Member 2 also need the corresponding forum text and structured metadata.

A naive merge such as `labels.merge(raw, on="unique_post_id")` is unsafe when the raw website export contains repeated IDs. It can:

- multiply rows,
- attach a label to the wrong content,
- silently change the dataset size,
- or leak duplicates into later splits.

## Matching strategy

The enrichment module streams the large corpus and uses `unique_post_id` as the primary key. If one label ID maps to more than one website row, normalized content is used as a second piece of evidence.

The rule is intentionally conservative:

- unique ID match → accept;
- duplicate ID + one exact normalized-content match → accept that row;
- duplicate ID + no unique content resolution → mark ambiguous and exclude rather than guess.

In the accepted project data:

- **5,617** canonical labels entered enrichment;
- **5,615** were safely matched;
- **19** duplicate-ID cases were resolved by exact content;
- **2** ambiguous duplicate-ID cases were excluded.

Excluding two uncertain rows is preferable to contaminating the evaluation foundation with an unverifiable match.

## Why stream the large CSV?

The website corpus is much larger than the labeled subset. Streaming lets the extraction inspect the corpus once without requiring every copy to remain in memory. It also makes the expensive raw-data dependency isolated to one stage.

When the large file is not bundled, the notebook restores the accepted frozen enrichment. That is a reproducibility convenience, not a claim that the raw scan ran again.


In [5]:
phase2_output = RUN_ROOT / "phase2"
phase2_enriched = phase2_output / "canonical_labels_enriched.csv.gz"
phase2_report_path = phase2_output / "phase2_report.json"
phase2_audit_path = phase2_output / "phase2_match_audit.csv"

raw_available = bool(RAW_360K_PATH and RAW_360K_PATH.exists())

if raw_available:
    print("Running the true streaming Phase 2 extraction.")
    run_command([
        PYTHON,
        ROOT / "src/phase2_extract_enriched.py",
        "--raw", RAW_360K_PATH,
        "--labels", phase1_output / "canonical_labels.csv.gz",
        "--output", phase2_enriched,
        "--audit", phase2_audit_path,
        "--report", phase2_report_path,
        # Strict mode is intentionally omitted because the project accepted two
        # unresolved ambiguous rows.
    ])
    phase2_mode = "full_raw_streaming"
else:
    if not USE_FROZEN_PHASE2_IF_RAW_MISSING:
        raise FileNotFoundError(
            "The 360k raw CSV is unavailable and cached Phase 2 mode is disabled."
        )
    print("Raw 360k CSV not found. Restoring the accepted frozen Phase 2 output.")
    shutil.copy2(
        ROOT / "frozen_outputs/canonical_labels_enriched.csv.gz",
        phase2_enriched,
    )
    shutil.copy2(
        ROOT / "frozen_outputs/phase2_report.json",
        phase2_report_path,
    )
    phase2_mode = "frozen_accepted_enrichment"

phase2_report = json.loads(phase2_report_path.read_text(encoding="utf-8"))
enriched = pd.read_csv(
    phase2_enriched,
    dtype={"unique_post_id": "string"},
)

print("Phase 2 mode:", phase2_mode)
display(pd.json_normalize(phase2_report))
print("Enriched rows:", len(enriched))
print("Unique enriched IDs:", enriched["unique_post_id"].nunique())
print("Raw/enriched columns:", len(enriched.columns))
display(enriched.head(3))

Raw 360k CSV not found. Restoring the accepted frozen Phase 2 output.


Phase 2 mode: frozen_accepted_enrichment


,raw_rows_scanned,canonical_labels_requested,enriched_rows_written,unresolved_rows,unresolved_ids,target_ids_with_duplicate_source_rows,strict_mode,note,match_status_counts.unique_id_match,match_status_counts.duplicate_id_resolved_exact_content,match_status_counts.ambiguous_duplicate_id_no_label_content
0,362017,5617,5615,2,"[397068790, thread_11018093_رانندگی_1402/02/15...",21,False,Frozen successful Phase 2 result. Two ambiguou...,5596,19,2


Enriched rows: 5615
Unique enriched IDs: 5615
Raw/enriched columns: 67


,thread_id,thread_title,author,posted_at,content,likes,reply_to,is_starter,user_post_count,gender,gender_code,join_date,children_count,category,sub_category,age_num,education_clean,unique_post_id,sig_char_count,sig_punct_count,sig_question_count,sig_excl_count,sig_emoji_count,sig_word_count,sig_neg_count,sig_pos_count,sig_pos_emoji,sig_neg_emoji,post_char_count,post_punct_count,post_question_count,post_excl_count,post_emoji_count,post_pos_emoji,post_neg_emoji,post_word_count,post_neg_count,post_pos_count,stress_proxy,post_neg_count_temp,post_pos_count_temp,label__unique_post_id,label__content,label__content_hash,label__final_stress,label__clinical_class,label__label_source,label__label_round,label__person1_raw,label__person2_raw,label__person1_estimated,label__historical_active_final,label__historical_split,label__historical_included,label__selection_method,label__confidence_tier,label__default_label_weight,label__duplicate_id_across_rounds,label__annotation_versions,label__discarded_label_sources,label__duplicate_content_group_size,label__exact_duplicate_content,label__notes,match_status,source_duplicate_count,matched_source_row_number,raw_content_hash
0,9125099,چند شبه جدا از شوهرم میخوابم😒,parvane53,1401/05/10 12:56,باید نزدیکش باشم نمیشه تنهاش گذاشت. مشکلای زیا...,2,246933654,False,208.0,زن,1,1401/01/27,1,دوران-بارداری,سلامت-عمومی-ورزش-تغذیه-تناسب-اندام-زیبایی-و-آ...,-1,نامشخص,246934094,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,73,7,0,0,1,0,1,16,1,0,2.50,0,0,246934094,باید نزدیکش باشم نمیشه تنهاش گذاشت. مشکلای زیا...,f71e666fe548534f995c6666bb9429f909dc1d03b6f45a...,4.111111,Moderate,active_dual,active_learning_round,3.0,6.0,NaN,4.111111,NaN,False,active_learning_threshold_near_boundary; 750 p...,high,1.00,False,1,NaN,1,False,NaN,unique_id_match,1,77620,f71e666fe548534f995c6666bb9429f909dc1d03b6f45a...
1,9116046,ورزش بارداری,mommy_narges,1401/05/14 01:38,من از پانزده هفته شروع کردم. شما هم هر زمان شر...,1,247322621,False,6330.0,زن,1,1398/08/15,0,دوران-بارداری,سلامت-عمومی-ورزش-تغذیه-تناسب-اندام-زیبایی-و-آ...,-1,نامشخص,247447668,91,13,0,0,0,24,0,0,0,0,158,5,0,0,0,0,0,36,2,5,0.30,0,1,247447668,من از پانزده هفته شروع کردم. شما هم هر زمان شر...,07f68178d2a618717af36938250c933796fcb9709528e8...,1.094246,Low,active_p2_calibrated,active_learning_round,NaN,1.0,1.149684,1.094246,NaN,False,active_learning_threshold_near_boundary; 750 p...,medium,0.65,False,1,NaN,1,False,NaN,unique_id_match,1,77490,07f68178d2a618717af36938250c933796fcb9709528e8...
2,9242694,بتا ۱۹۷۳۱ یعنی هفته چندمه حاملگیه؟؟؟,عشق5,1401/05/28 17:15,من بتام 39999 بود هفت هفته بودم\nگفت چن قلو هس...,0,-1,False,8869.0,زن,1,1398/04/04,0,دوران-بارداری,سلامت-عمومی-ورزش-تغذیه-تناسب-اندام-زیبایی-و-آ...,40,فوق لیسانس,249384092,28,6,0,0,0,7,0,0,0,0,195,1,0,0,0,0,0,44,3,2,1.34,0,0,249384092,من بتام 39999 بود هفت هفته بودم\nگفت چن قلو هس...,03d4e07dececb353b0b253d50a702fc87719ff019a9f9c...,2.629630,Low,active_dual,active_learning_round,3.0,2.0,NaN,2.629630,NaN,False,active_learning_threshold_near_boundary; 750 p...,high,1.00,False,1,NaN,1,False,NaN,unique_id_match,1,76876,03d4e07dececb353b0b253d50a702fc87719ff019a9f9c...


**What a successful Phase 2 result means**

The enriched table should contain exactly **5,615 unique IDs** and the metadata/text needed by the later grouping and model handoffs.

The two excluded ambiguous rows are a deliberate quality-control decision. They should not be reintroduced merely to make the count equal 5,617.

If cached mode is shown, the correct interpretation is: *the accepted enrichment artifact was restored and downstream logic was fully rerun*. If the raw corpus is present, the report/audit additionally demonstrates that the matching itself was recomputed.


# Historical Phase 3 — Build an immutable leakage-safe grouped split

A normal random row split assumes observations are independent. Forum posts violate that assumption.

Two posts can be statistically related because they:

- were written by the **same author**,
- occur in the **same thread/conversation**,
- or have **exact duplicate normalized content**.

If one related row is in training and another in validation/test, a model can partially recognize the person, conversation, or copied text. The measured error would then exaggerate generalization to genuinely new contexts.

## Connected-component construction

The split builder treats the dataset as a graph.

For each post, it creates links to nodes representing meaningful author IDs, thread IDs, and repeated content hashes. It uses a **Union-Find / Disjoint Set Union** data structure to merge all transitively connected rows.

The transitive part is important. Suppose:

- post A shares an author with post B;
- post B shares a thread with post C.

Even if A and C share no field directly, they belong to the same connected component and must remain together. Union-Find efficiently discovers these components using path compression and union by rank.

The final enriched data form **3,622 connected components**.

## Component-level split optimization

Whole components, not individual rows, are assigned to train/validation/test.

Because component sizes and rare-class composition vary, exact 70/15/15 proportions cannot simply be obtained by shuffling. The script uses a randomized greedy assignment with multiple restarts. Its objective penalizes deviation from:

- desired total split sizes,
- per-class evaluation counts,
- annotation-source composition,
- and unnecessary training-only labels inside validation/test components.

Rare/class-informative groups are considered early, and **300 deterministic restarts** explore different tie orders. The best objective value is kept.

This is an engineering optimization, not a statistical guarantee of a unique globally optimal partition. The scientific requirement is stronger and simpler: **no connected component may cross a split boundary**.

## Evaluation eligibility

Only the strongest dual-annotation sources are official evaluation candidates by default. Weaker single-rater/calibrated rows can still exist inside the same connected component. Phase 3.1 resolves what to do with those rows without violating the component boundary.


In [6]:
phase3_output = RUN_ROOT / "phase3"
split_manifest_path = phase3_output / "split_manifest.csv"
phase3_report_path = phase3_output / "phase3_split_report.json"

run_command([
    PYTHON,
    ROOT / "src/phase3_build_split_manifest.py",
    "--enriched", phase2_enriched,
    "--output", split_manifest_path,
    "--report", phase3_report_path,
    "--seed", PHASE3_SEED,
    "--restarts", PHASE3_RESTARTS,
    "--min-eval-per-class", 25,
])

phase3_report = json.loads(phase3_report_path.read_text(encoding="utf-8"))
split_manifest = pd.read_csv(
    split_manifest_path,
    dtype={"unique_post_id": "string"},
)

display(pd.json_normalize(phase3_report))
display(
    split_manifest.groupby(["split", "clinical_class"])
    .size()
    .unstack(fill_value=0)
)
print("Rows:", len(split_manifest))
print("Connected components:", split_manifest["group_id"].nunique())
print(
    "Maximum number of splits occupied by one component:",
    split_manifest.groupby("group_id")["split"].nunique().max(),
)


$ /opt/pyvenv/bin/python /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/src/phase3_build_split_manifest.py --enriched /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase2/canonical_labels_enriched.csv.gz --output /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase3/split_manifest.csv --report /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase3/phase3_split_report.json --seed 20260804 --restarts 300 --min-eval-per-class 25


{
  "status": "complete",
  "dataset_hash": "f9b1810e611cf7b7c25ea3e15c4f4145a1df7b92435c2953b72e088a89f7e109",
  "seed": 20260804,
  "ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "rows": 5615,
  "connected_components": 3622,
  "largest_component_sizes": [
    63,
    42,
    28,
    24,
    17,
    15,
    13,
    13,
    12,
    12,
    12,
    11,
    10,
    10,
    10,
    9,
    9,
    9,
    9,
    9
  ],
  "assignment_objective": 0.41640301194364504,
  "split_counts": {
    "train": 4106,
    "val": 754,
    "test": 755
  },
  "evaluation_sources": [
    "active_dual",
    "original_dual"
  ],
  "evaluation_class_counts": {
    "train": {
      "Moderate": 311,
      "Low": 1377,
      "High": 291,
      "Very High": 116
    },
    "val": {
      "Moderate": 67,
      "Low": 297,
      "High": 63,
      "Very High": 25
    },
    "test": {
      "Low": 297,
      "High": 63,
      "Very High": 26,
      "Moderate": 67
    }
  },
  "label_source_counts":

,status,dataset_hash,seed,rows,connected_components,largest_component_sizes,assignment_objective,evaluation_sources,minimum_eval_per_class,minimum_failures,leakage_guards,evaluation_scope,ratios.train,ratios.val,ratios.test,split_counts.train,split_counts.val,split_counts.test,evaluation_class_counts.train.Moderate,evaluation_class_counts.train.Low,evaluation_class_counts.train.High,evaluation_class_counts.train.Very High,evaluation_class_counts.val.Moderate,evaluation_class_counts.val.Low,evaluation_class_counts.val.High,evaluation_class_counts.val.Very High,evaluation_class_counts.test.Low,evaluation_class_counts.test.High,evaluation_class_counts.test.Very High,evaluation_class_counts.test.Moderate,label_source_counts.train.active_dual,label_source_counts.train.active_p2_calibrated,label_source_counts.train.original_p1_high_selected,label_source_counts.train.original_dual,label_source_counts.val.active_p2_calibrated,label_source_counts.val.active_dual,label_source_counts.val.original_p1_high_selected,label_source_counts.val.original_dual,label_source_counts.test.active_p2_calibrated,label_source_counts.test.active_dual,label_source_counts.test.original_p1_high_selected,label_source_counts.test.original_dual
0,complete,f9b1810e611cf7b7c25ea3e15c4f4145a1df7b92435c29...,20260804,5615,3622,"[63, 42, 28, 24, 17, 15, 13, 13, 12, 12, 12, 1...",0.416403,"[active_dual, original_dual]",25,[],"[author connected components, thread connected...",safety-enriched labeled data; not a natural-pr...,0.7,0.15,0.15,4106,754,755,311,1377,291,116,67,297,63,25,297,63,26,67,699,1561,450,1396,217,150,85,302,217,151,85,302


clinical_class,High,Low,Moderate,Very High
split,,,,
test,135,413,159,48
train,706,2205,921,274
val,132,411,141,70


Rows: 5615
Connected components: 3622
Maximum number of splits occupied by one component: 1


**How to interpret the Phase 3 output**

The most important leakage assertion is:

> maximum number of splits occupied by one connected component = 1.

Class-count balance matters, but leakage prevention takes priority over perfectly matching percentages.

The initial component split is intentionally not yet the final model-role table. Some validation/test components may contain both official evaluation rows and weaker training-only labels. Sending those weaker rows back into training would leak information from an evaluation-connected component. Phase 3.1 therefore introduces the explicit `embargo` role.


# Historical Phase 3.1 — Final model roles, OOF folds, and training weights

This stage converts the component split into the exact contract consumed by Members 1 and 2.

## 1. Why four roles instead of only train/validation/test?

- **Train:** rows allowed to fit model parameters.
- **Validation:** official dual-labeled rows used for model/candidate selection and threshold calibration.
- **Test:** official dual-labeled rows opened only after the final model/thresholds were frozen.
- **Embargo:** non-official rows that belong to a validation/test connected component.

Embargo is a leakage barrier. A row can be perfectly usable text but still be inappropriate for training because it is connected to an evaluation row.

If a component originally assigned to validation/test contains **no official evaluation row at all**, the entire component can safely move to training. Otherwise, only official rows remain in evaluation and connected non-official rows become embargo.

This cleanup produces the final **4,226 / 452 / 453 / 484** roles.

## 2. Five grouped OOF folds

The 4,226 training rows are split with `StratifiedGroupKFold`.

- **Grouping by `group_id`** keeps every connected component in one fold.
- **Stratification by `clinical_class`** tries to preserve the ordered-class distribution across folds.
- Five folds give each base model ~80% of training data while still generating one held-out prediction for every training row.

The final fold sizes are **844, 844, 848, 847, 843**.

These folds are critical for later stacking. Member 1 and Member 2 must generate **out-of-fold (OOF)** predictions so that the fusion model never trains on a base-model score produced by a model that already saw that row.

## 3. Training weights

Two ideas are multiplied:

### Annotation-confidence weight

`annotation_weight` reflects provenance. Stronger labels can have more influence than lower-confidence calibrated labels.

### Square-root inverse-frequency class weight

For class \(c\) with count \(n_c\), the project uses a moderate class-balancing factor proportional to:

\[
w_c = \sqrt{\frac{N}{K\,n_c}}
\]

where \(N\) is the number of training rows and \(K=4\) classes.

Why the square root? Full inverse-frequency weighting can make the rare Very High class dominate a small dataset and increase variance. Square-root weighting gives minority classes more influence without making the correction as aggressive.

The product of annotation and class weights is normalized so the **mean training sample weight is 1**. Validation/test metrics remain unweighted so reported performance describes actual held-out rows.

## 4. Member-specific handoffs

The same role/fold/target contract is exported to both members:

- Member 1 receives structured metadata and predefined count features.
- Member 2 receives raw `content` as the only predictive modality.

This common foundation is what makes their later predictions align safely by `unique_post_id`.


In [7]:
phase3_1_output = RUN_ROOT / "phase3_1"

run_command([
    PYTHON,
    ROOT / "src/phase3_1_finalize_modeling_manifest.py",
    "--enriched", phase2_enriched,
    "--manifest", split_manifest_path,
    "--phase3-report", phase3_report_path,
    "--output-dir", phase3_1_output,
    "--folds", 5,
])

modeling_manifest = pd.read_csv(
    phase3_1_output / "modeling_manifest_v2.csv",
    dtype={"unique_post_id": "string"},
)
cleanup_report = json.loads(
    (phase3_1_output / "cleanup_report.json").read_text(encoding="utf-8")
)

display(pd.json_normalize(cleanup_report))
display(
    modeling_manifest.groupby(["model_role", "clinical_class"])
    .size()
    .unstack(fill_value=0)
)
display(pd.read_csv(phase3_1_output / "oof_fold_distribution.csv"))
display(pd.read_csv(phase3_1_output / "acceptance_counts.csv"))


$ /opt/pyvenv/bin/python /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/src/phase3_1_finalize_modeling_manifest.py --enriched /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase2/canonical_labels_enriched.csv.gz --manifest /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase3/split_manifest.csv --phase3-report /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase3/phase3_split_report.json --output-dir /mnt/data/memberC_reproducible_notebooks/MemberC_Phases_1_3/notebook_run/phase3_1 --folds 5


{
  "status": "complete",
  "source_dataset_hash": "f9b1810e611cf7b7c25ea3e15c4f4145a1df7b92435c2953b72e088a89f7e109",
  "seed": 20260804,
  "rows_total": 5615,
  "roles": {
    "train": 4226,
    "embargo": 484,
    "test": 453,
    "validation": 452
  },
  "rows_moved_from_non_eval_val_test_components_to_train": 120,
  "embargo_rows": 484,
  "oof_folds": 5,
  "training_class_counts": {
    "Low": 2257,
    "Moderate": 948,
    "High": 736,
    "Very High": 285
  },
  "class_weight_method": "square-root inverse frequency",
  "class_weight_map": {
    "Low": 0.684177788873456,
    "Moderate": 1.0556758388791745,
    "High": 1.198107656482396,
    "Very High": 1.9253616657292336
  },
  "training_sample_weight_mean": 1.0000000000000004
}

Spreadsheet runtime warmup failed during python startup
Traceback (most recent call last):
  File "/tmp/tmp.yTcnQsZYiA/artifact_tool_v2-2.8.4/artifact_tool/patches/warm_spreadsheet_runtime_on_startup.py", line 26, in warm_spreadsheet_runtime_on_startup


,status,source_dataset_hash,seed,rows_total,rows_moved_from_non_eval_val_test_components_to_train,embargo_rows,oof_folds,class_weight_method,training_sample_weight_mean,roles.train,roles.embargo,roles.test,roles.validation,training_class_counts.Low,training_class_counts.Moderate,training_class_counts.High,training_class_counts.Very High,class_weight_map.Low,class_weight_map.Moderate,class_weight_map.High,class_weight_map.Very High
0,complete,f9b1810e611cf7b7c25ea3e15c4f4145a1df7b92435c29...,20260804,5615,120,484,5,square-root inverse frequency,1.0,4226,484,453,452,2257,948,736,285,0.684178,1.055676,1.198108,1.925362


clinical_class,High,Low,Moderate,Very High
model_role,,,,
embargo,111,178,139,56
test,63,297,67,26
train,736,2257,948,285
validation,63,297,67,25


,oof_fold,High,Low,Moderate,Very High
0,0,147,451,189,57
1,1,147,451,189,57
2,2,148,452,191,57
3,3,148,452,190,57
4,4,146,451,189,57


,evaluation_split,clinical_class,available_rows,minimum_recall,minimum_correct_predictions,maximum_allowed_misses
0,validation,Low,297,0.75,223,74
1,validation,Moderate,67,0.50,34,33
2,validation,High,63,0.50,32,31
3,validation,Very High,25,0.75,19,6
4,test,Low,297,0.75,223,74
5,test,Moderate,67,0.50,34,33
6,test,High,63,0.50,32,31
7,test,Very High,26,0.75,20,6


**Final Part 2 modeling contract**

A correct run must produce:

| Role | Rows | Purpose |
|---|---:|---|
| Train | 4,226 | Fit Member 1, Member 2, and later fusion |
| Validation | 452 | Model/candidate selection and threshold calibration |
| Locked test | 453 | One-time final evidence |
| Embargo | 484 | Excluded because of evaluation-component/source constraints |

The five grouped training folds contain **844 / 844 / 848 / 847 / 843** rows.

The evaluation set is intentionally enriched with informative/stressful examples. Therefore these counts and class proportions are useful for **model evaluation**, but they do not estimate how common stress is in the natural website stream.


# Final validation and reproducibility manifest

The last code block turns the methodological assumptions into executable assertions.

It checks that:

- all **5,615** rows are present exactly once;
- role counts equal the frozen contract;
- every training row has an OOF fold;
- non-training rows have no training fold;
- no connected component crosses OOF folds.

These assertions are more valuable than simply printing a table: if an invariant is violated, execution stops.

The notebook also performs artifact-integrity diagnostics as part of reproduction. These checks are supplementary to the scientific assertions and do not require separate checksum manifest files.


In [8]:
expected_roles = {
    "train": 4226,
    "validation": 452,
    "test": 453,
    "embargo": 484,
}
actual_roles = modeling_manifest["model_role"].value_counts().to_dict()

assert len(modeling_manifest) == 5615
assert actual_roles == expected_roles, (actual_roles, expected_roles)
assert modeling_manifest["unique_post_id"].nunique() == 5615
assert (
    modeling_manifest.loc[modeling_manifest["model_role"].eq("train"), "oof_fold"]
    .notna()
    .all()
)
assert (
    modeling_manifest.loc[modeling_manifest["model_role"].ne("train"), "oof_fold"]
    .isna()
    .all()
)
assert (
    modeling_manifest.loc[modeling_manifest["model_role"].eq("train")]
    .groupby("group_id")["oof_fold"]
    .nunique()
    .max()
    == 1
)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

tracked = [
    phase1_output / "canonical_labels.csv.gz",
    phase1_output / "annotation_history.csv.gz",
    phase2_enriched,
    split_manifest_path,
    phase3_1_output / "modeling_manifest_v2.csv",
    phase3_1_output / "member1_handoff.csv.gz",
    phase3_1_output / "member2_handoff.csv.gz",
]
checksums = {
    str(path.relative_to(RUN_ROOT)): sha256_file(path)
    for path in tracked
}
(RUN_ROOT / "checksums.json").write_text(
    json.dumps(checksums, indent=2),
    encoding="utf-8",
)

print("All Phase 1–3 invariants passed.")
display(pd.DataFrame(
    [{"artifact": key, "sha256": value} for key, value in checksums.items()]
))

All Phase 1–3 invariants passed.


,artifact,sha256
0,phase1/canonical_labels.csv.gz,86680e8a448d57e3527c57877c16e694877270ec87aa0e...
1,phase1/annotation_history.csv.gz,eef1dfd52cc78d2be107b6fd3f069de0c8a6b9d0e6ebd9...
2,phase2/canonical_labels_enriched.csv.gz,1f7a4a18924cb9003c7fa9459a85d1c822166ceb81ed10...
3,phase3/split_manifest.csv,3c809baddd69148a428c767c9680a0e1c268538807ab73...
4,phase3_1/modeling_manifest_v2.csv,177327ab5861a79a69f1cb3f539d30234d921c6b7e662b...
5,phase3_1/member1_handoff.csv.gz,d426386a4aab80df1a072afc308e77efc1cfe710d876fd...
6,phase3_1/member2_handoff.csv.gz,6b99feda23226702891d2c881149f58b0527f6ff2fb110...


# Part 2 completion — what we learned and what is handed to Part 3

Part 2 establishes the data foundation on which every later performance claim depends.

The methodological path is:

**multi-round annotations → provenance-aware canonical labels → duplicate-safe website enrichment → author/thread/content connected components → component-level split → official evaluation eligibility → embargo protection → grouped stratified OOF folds → confidence/class training weights → Member 1 and Member 2 handoffs**

Principal outputs:

- `notebook_run/phase1/canonical_labels.csv.gz`
- `notebook_run/phase2/canonical_labels_enriched.csv.gz`
- `notebook_run/phase3/split_manifest.csv`
- `notebook_run/phase3_1/modeling_manifest_v2.csv`
- `notebook_run/phase3_1/member1_handoff.csv.gz`
- `notebook_run/phase3_1/member2_handoff.csv.gz`

## Scientific interpretation

A later model can still make mistakes, but it should not obtain an artificially easy validation/test result by seeing the same author, thread, exact duplicate text, or in-sample base prediction across the boundary.

That is the main lesson of this notebook: **experimental design and leakage control are part of the machine-learning algorithmic pipeline, not an afterthought.**

For a strict re-extraction of Phase 2, provide the external cleaned website CSV and rerun the notebook. Otherwise the accepted frozen enrichment is restored transparently.
